# Layer-aware FD-loss on MNIST

This notebook adapts the FD-loss idea from *Representation Frechet Loss for Visual Generation* (arXiv:2604.28190) to a small MNIST experiment.

The paper's important training trick is to decouple the population used to estimate feature-distribution statistics from the batch used for gradient computation. Here we do the same with an EMA of generated feature moments, but we make the representation space layer-aware:

```text
L_layer = sum_l lambda_l FD(phi_l(x_real), phi_l(x_fake))
```

For this MNIST-scale classifier, the layers are:

- `early`: channel mean/std after the first conv block, mostly stroke texture and local ink statistics.
- `middle`: channel mean/std after the second conv block, more digit parts and geometry.
- `late`: the 128-d penultimate representation, more class semantics.

The layer weights are explicit knobs in `LAYER_WEIGHTS`.

In [ ]:
from pathlib import Path
import struct
import gzip
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MNIST_DIR = Path(os.environ.get("MNIST_DIR", "/kaggle/input/datasets/hojjatk/mnist-dataset"))
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
WORK_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 10
TRAIN_BATCH = 256
TEST_BATCH = 512


def resolve_idx_file(path: Path) -> Path:
    """Kaggle sometimes exposes IDX files as directories; find the real file."""
    path = Path(path)

    if path.is_file():
        return path

    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file()]
        if len(files) == 1:
            return files[0]
        if len(files) > 1:
            for p in files:
                if path.name in p.name:
                    return p
            return files[0]

    raise FileNotFoundError(f"Could not resolve IDX file from: {path}")


def read_idx_images(path):
    path = resolve_idx_file(Path(path))
    opener = gzip.open if path.suffix == ".gz" else open

    with opener(path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        assert magic == 2051, f"Bad image magic number: {magic}"
        data = np.frombuffer(f.read(), dtype=np.uint8)

    return torch.tensor(data.copy(), dtype=torch.float32).view(n, 1, rows, cols) / 255.0


def read_idx_labels(path):
    path = resolve_idx_file(Path(path))
    opener = gzip.open if path.suffix == ".gz" else open

    with opener(path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        assert magic == 2049, f"Bad label magic number: {magic}"
        data = np.frombuffer(f.read(), dtype=np.uint8)

    return torch.tensor(data.copy(), dtype=torch.long)


x_train = read_idx_images(MNIST_DIR / "train-images-idx3-ubyte")
y_train = read_idx_labels(MNIST_DIR / "train-labels-idx1-ubyte")
x_test = read_idx_images(MNIST_DIR / "t10k-images-idx3-ubyte")
y_test = read_idx_labels(MNIST_DIR / "t10k-labels-idx1-ubyte")

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=TRAIN_BATCH, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=TEST_BATCH, shuffle=False)

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape, DEVICE)

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.10)
        self.head = nn.Linear(128, 10)

    @staticmethod
    def channel_mean_std(h):
        mean = h.mean(dim=(2, 3))
        std = h.std(dim=(2, 3), unbiased=False)
        return torch.cat([mean, std], dim=1)

    def forward_features(self, x):
        h1 = F.relu(self.conv1(x))
        p1 = F.max_pool2d(h1, 2)

        h2 = F.relu(self.conv2(p1))
        p2 = F.max_pool2d(h2, 2)

        flat = p2.flatten(1)
        late = F.relu(self.fc1(flat))

        return {
            "early": self.channel_mean_std(p1),
            "middle": self.channel_mean_std(p2),
            "late": late,
        }

    def forward(self, x):
        feats = self.forward_features(x)
        return self.head(self.dropout(feats["late"]))


def augment_mnist_batch(x):
    """Fast GPU-friendly MNIST augmentation with small translations and mild noise."""
    shifts_y = torch.randint(-2, 3, (x.size(0),), device=x.device)
    shifts_x = torch.randint(-2, 3, (x.size(0),), device=x.device)
    x_aug = x.clone()

    for i in range(x.size(0)):
        x_aug[i] = torch.roll(x_aug[i], shifts=(int(shifts_y[i]), int(shifts_x[i])), dims=(1, 2))

    x_aug = x_aug + 0.03 * torch.randn_like(x_aug)
    return x_aug.clamp(0.0, 1.0)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)

    return total_loss / total, correct / total


classifier = MNISTClassifier().to(DEVICE)
CKPT_PATH = WORK_DIR / "mnist_layer_feature_classifier.pt"
print(classifier)

In [ ]:
EPOCHS = 10
LR = 1e-3

if CKPT_PATH.exists():
    classifier.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    print("Loaded classifier checkpoint.")
else:
    print("No checkpoint found; training classifier with augmentation...")
    opt = torch.optim.AdamW(classifier.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        classifier.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = classifier(augment_mnist_batch(xb))
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        test_loss, test_acc = evaluate(classifier, test_loader)
        print(
            f"epoch {epoch + 1}/{EPOCHS} | "
            f"train loss={train_loss:.4f}, train acc={train_acc:.4f} | "
            f"test loss={test_loss:.4f}, test acc={test_acc:.4f}"
        )

    torch.save(classifier.state_dict(), CKPT_PATH)
    print(f"Saved checkpoint to {CKPT_PATH}")

classifier.eval()
for p in classifier.parameters():
    p.requires_grad_(False)

test_loss, test_acc = evaluate(classifier, test_loader)
print(f"final test accuracy: {test_acc:.4f}")

In [ ]:
class LayerFeatureExtractor(nn.Module):
    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier

    def forward(self, x):
        return self.classifier.forward_features(x)


phi = LayerFeatureExtractor(classifier).to(DEVICE)
phi.eval()
for p in phi.parameters():
    p.requires_grad_(False)

with torch.no_grad():
    sample_features = phi(x_train[:8].to(DEVICE))

LAYER_DIMS = {name: feats.shape[1] for name, feats in sample_features.items()}
LAYER_NAMES = list(LAYER_DIMS.keys())

print("Layer feature dimensions:")
for name, dim in LAYER_DIMS.items():
    print(f"  {name}: {dim}")

In [ ]:
@torch.no_grad()
def collect_layer_features_by_class(feature_model, loader, num_classes=NUM_CLASSES):
    buckets = {name: [[] for _ in range(num_classes)] for name in LAYER_NAMES}
    feature_model.eval()

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        features = feature_model(xb)

        for name, feats in features.items():
            feats_cpu = feats.detach().cpu()
            for digit in range(num_classes):
                mask = (yb == digit)
                if mask.any():
                    buckets[name][digit].append(feats_cpu[mask])

    return {
        name: [torch.cat(buckets[name][digit], dim=0) for digit in range(num_classes)]
        for name in LAYER_NAMES
    }


def mean_cov_from_features(feats, eps=1e-4):
    mu = feats.mean(dim=0)
    centered = feats - mu
    cov = centered.T @ centered / (feats.shape[0] - 1)
    cov = 0.5 * (cov + cov.T)
    cov = cov + eps * torch.eye(cov.shape[0])
    return mu, cov


features_by_layer_digit = collect_layer_features_by_class(phi, train_loader)
real_stats = {}

for name in LAYER_NAMES:
    mus = []
    covs = []

    print(f"\n{name} features")
    for digit, feats in enumerate(features_by_layer_digit[name]):
        mu_y, cov_y = mean_cov_from_features(feats)
        mus.append(mu_y)
        covs.append(cov_y)
        print(f"  digit {digit}: n={feats.shape[0]}, dim={feats.shape[1]}")

    real_stats[name] = {
        "mu": torch.stack(mus).to(DEVICE),
        "cov": torch.stack(covs).to(DEVICE),
        "dim": LAYER_DIMS[name],
    }

print("\nReal statistics ready:")
for name, stats in real_stats.items():
    print(name, stats["mu"].shape, stats["cov"].shape)

In [ ]:
@torch.no_grad()
def sqrtm_psd_nograd(a, eps=1e-8):
    a = 0.5 * (a + a.transpose(-1, -2))
    eigvals, eigvecs = torch.linalg.eigh(a)
    eigvals = eigvals.clamp_min(eps)
    return (eigvecs * eigvals.sqrt().unsqueeze(-2)) @ eigvecs.transpose(-1, -2)


for name, stats in real_stats.items():
    stats["cov_sqrt"] = torch.stack([
        sqrtm_psd_nograd(stats["cov"][digit])
        for digit in range(NUM_CLASSES)
    ])


STATS_PATH = WORK_DIR / "mnist_layer_feature_stats.pt"
torch.save(
    {
        "layer_names": LAYER_NAMES,
        "layer_dims": LAYER_DIMS,
        "real_stats": {
            name: {
                "mu": stats["mu"].cpu(),
                "cov": stats["cov"].cpu(),
            }
            for name, stats in real_stats.items()
        },
    },
    STATS_PATH,
)
print(f"saved layer feature stats to {STATS_PATH}")

In [ ]:
def plot_covariance_matrix(stats, layer="late", digit=0, title_prefix="Real"):
    cov = stats[layer]["cov"][digit].detach().cpu().numpy()
    plt.figure(figsize=(6, 5))
    plt.imshow(cov, cmap="viridis")
    plt.colorbar()
    plt.title(f"{title_prefix} covariance | layer={layer}, digit={digit}")
    plt.xlabel("feature index")
    plt.ylabel("feature index")
    plt.tight_layout()
    plt.show()


plot_covariance_matrix(real_stats, layer="early", digit=0)
plot_covariance_matrix(real_stats, layer="middle", digit=0)
plot_covariance_matrix(real_stats, layer="late", digit=0)

In [ ]:
class OneStepGenerator(nn.Module):
    def __init__(self, z_dim=64, y_dim=10):
        super().__init__()
        self.z_dim = z_dim
        self.y_emb = nn.Embedding(y_dim, 16)
        self.net = nn.Sequential(
            nn.Linear(z_dim + 16, 256),
            nn.SiLU(),
            nn.Linear(256, 512),
            nn.SiLU(),
            nn.Linear(512, 1024),
            nn.SiLU(),
            nn.Linear(1024, 28 * 28),
        )

    def forward(self, z, y):
        y_e = self.y_emb(y)
        h = torch.cat([z, y_e], dim=1)
        x = torch.sigmoid(self.net(h))
        return x.view(-1, 1, 28, 28)


def fd_to_real_layer(mu_g, cov_g, layer, digit):
    """FD between generated stats and fixed real stats for one layer and digit."""
    mu_r = real_stats[layer]["mu"][digit]
    cov_r = real_stats[layer]["cov"][digit]
    cov_r_sqrt = real_stats[layer]["cov_sqrt"][digit]

    mean_term = (mu_g - mu_r).pow(2).sum()
    prod = cov_r_sqrt @ cov_g @ cov_r_sqrt
    prod = 0.5 * (prod + prod.T)
    eigvals = torch.linalg.eigvalsh(prod).clamp_min(1e-8)
    trace_sqrt = eigvals.sqrt().sum()
    cov_term = torch.trace(cov_g) + torch.trace(cov_r) - 2.0 * trace_sqrt
    return mean_term + cov_term


def batch_moments_by_digit(feats, y, dim):
    mus_b = []
    M2s_b = []

    for digit in range(NUM_CLASSES):
        f = feats[y == digit]
        if f.shape[0] == 0:
            mus_b.append(torch.zeros(dim, device=DEVICE))
            M2s_b.append(torch.zeros(dim, dim, device=DEVICE))
            continue

        mus_b.append(f.mean(dim=0))
        M2s_b.append(f.T @ f / f.shape[0])

    return torch.stack(mus_b), torch.stack(M2s_b)


def cov_from_raw_moments(mu, M2, eps=1e-4):
    dim = mu.shape[0]
    cov = M2 - torch.outer(mu, mu)
    cov = 0.5 * (cov + cov.T)
    return cov + eps * torch.eye(dim, device=mu.device)


G = OneStepGenerator(z_dim=64).to(DEVICE)
print(G)

In [ ]:
z_dim = 64
EPS = 1e-4

# These are the layer-aware knobs. Increase early for texture/stroke statistics,
# middle for parts/geometry, and late for class semantics.
LAYER_WEIGHTS = {
    "early": 0.35,
    "middle": 0.65,
    "late": 1.00,
}


@torch.no_grad()
def init_generated_ema(G, rounds=32, per_digit=64):
    G.eval()
    buckets = {name: [[] for _ in range(NUM_CLASSES)] for name in LAYER_NAMES}

    for _ in range(rounds):
        y = torch.arange(NUM_CLASSES, device=DEVICE).repeat_interleave(per_digit)
        z = torch.randn(y.shape[0], z_dim, device=DEVICE)
        x_fake = G(z, y)
        features = phi(x_fake)

        for name, feats in features.items():
            for digit in range(NUM_CLASSES):
                buckets[name][digit].append(feats[y == digit].detach())

    ema = {}
    for name in LAYER_NAMES:
        dim = LAYER_DIMS[name]
        ema_mu = torch.zeros(NUM_CLASSES, dim, device=DEVICE)
        ema_M2 = torch.zeros(NUM_CLASSES, dim, dim, device=DEVICE)

        for digit in range(NUM_CLASSES):
            f = torch.cat(buckets[name][digit], dim=0)
            ema_mu[digit] = f.mean(dim=0)
            ema_M2[digit] = f.T @ f / f.shape[0]

        ema[name] = {"mu": ema_mu, "M2": ema_M2}

    return ema


ema_stats = init_generated_ema(G, rounds=32, per_digit=64)

print("Initialized generated EMA stats:")
for name in LAYER_NAMES:
    print(name, ema_stats[name]["mu"].shape, ema_stats[name]["M2"].shape)

In [ ]:
def total_variation_loss(x):
    tv_h = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    tv_w = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    return tv_h + tv_w


def foreground_mass_loss(x, target_mean=0.13):
    return (x.mean() - target_mean).pow(2)


def border_loss(x, border=3):
    top = x[:, :, :border, :].mean()
    bottom = x[:, :, -border:, :].mean()
    left = x[:, :, :, :border].mean()
    right = x[:, :, :, -border:].mean()
    return top + bottom + left + right


def lerp(a, b, t):
    return a + (b - a) * t

In [ ]:
G_STEPS = 3000
PER_DIGIT = 64
LR_G = 1e-3
PRINT_EVERY = 100
CE_WARMUP_STEPS = 500
BETA_START = 0.90
BETA_END = 0.97

opt_G = torch.optim.AdamW(G.parameters(), lr=LR_G, weight_decay=0.0)

for step in range(1, G_STEPS + 1):
    G.train()
    progress = step / G_STEPS
    beta = lerp(BETA_START, BETA_END, progress)

    y = torch.arange(NUM_CLASSES, device=DEVICE).repeat_interleave(PER_DIGIT)
    z = torch.randn(y.shape[0], z_dim, device=DEVICE)
    x_fake = G(z, y)

    features = phi(x_fake)
    logits = classifier(x_fake)

    fd_weighted = 0.0
    fd_raw_total = 0.0
    fd_layer_logs = {}
    new_ema = {}

    for layer in LAYER_NAMES:
        dim = LAYER_DIMS[layer]
        mu_b, M2_b = batch_moments_by_digit(features[layer], y, dim)
        layer_fd_raw = 0.0
        layer_fd_norm = 0.0
        new_mu = []
        new_M2 = []

        for digit in range(NUM_CLASSES):
            # Differentiable EMA estimate: old EMA is detached, current batch carries gradient.
            mu_g = beta * ema_stats[layer]["mu"][digit].detach() + (1.0 - beta) * mu_b[digit]
            M2_g = beta * ema_stats[layer]["M2"][digit].detach() + (1.0 - beta) * M2_b[digit]
            cov_g = cov_from_raw_moments(mu_g, M2_g, eps=EPS)
            fd_y = fd_to_real_layer(mu_g, cov_g, layer, digit)

            layer_fd_raw = layer_fd_raw + fd_y
            layer_fd_norm = layer_fd_norm + fd_y / fd_y.detach().clamp_min(1e-6)
            new_mu.append(mu_g.detach())
            new_M2.append(M2_g.detach())

        layer_fd_raw = layer_fd_raw / NUM_CLASSES
        layer_fd_norm = layer_fd_norm / NUM_CLASSES
        fd_layer_logs[layer] = layer_fd_raw.detach()
        fd_raw_total = fd_raw_total + layer_fd_raw.detach()
        fd_weighted = fd_weighted + LAYER_WEIGHTS[layer] * layer_fd_norm
        new_ema[layer] = {"mu": torch.stack(new_mu), "M2": torch.stack(new_M2)}

    ce_loss = F.cross_entropy(logits, y)
    tv_loss = total_variation_loss(x_fake)
    mass_loss = foreground_mass_loss(x_fake, target_mean=0.13)
    edge_loss = border_loss(x_fake, border=3)

    if step <= CE_WARMUP_STEPS:
        lambda_ce = 2.0
        lambda_fd = 0.25
        lambda_tv = 0.10
        lambda_mass = 2.0
        lambda_edge = 0.50
    else:
        lambda_ce = 0.15
        lambda_fd = 1.0
        lambda_tv = 0.15
        lambda_mass = 2.0
        lambda_edge = 0.50

    loss = (
        lambda_fd * fd_weighted
        + lambda_ce * ce_loss
        + lambda_tv * tv_loss
        + lambda_mass * mass_loss
        + lambda_edge * edge_loss
    )

    opt_G.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=5.0)
    opt_G.step()

    with torch.no_grad():
        ema_stats = new_ema

    if step % PRINT_EVERY == 0 or step == 1:
        pred_acc = (logits.argmax(dim=1) == y).float().mean().item()
        layer_log = " | ".join(f"FD_{k}={v.item():.2f}" for k, v in fd_layer_logs.items())
        print(
            f"step {step:04d} | "
            f"loss={loss.item():.4f} | "
            f"FD_weighted={fd_weighted.item():.4f} | "
            f"{layer_log} | "
            f"CE={ce_loss.item():.4f} | "
            f"TV={tv_loss.item():.4f} | "
            f"mass={mass_loss.item():.4f} | "
            f"edge={edge_loss.item():.4f} | "
            f"clf_acc={pred_acc:.3f} | "
            f"beta={beta:.3f} | "
            f"grad={float(grad_norm):.4f} | "
            f"x_mean={x_fake.mean().item():.3f} | "
            f"x_std={x_fake.std().item():.3f}"
        )

In [ ]:
@torch.no_grad()
def sample_digit_grid(G, z_dim=64, n_per_digit=10, seed=None):
    G.eval()
    if seed is not None:
        torch.manual_seed(seed)

    rows = []
    for digit in range(NUM_CLASSES):
        y = torch.full((n_per_digit,), digit, device=DEVICE, dtype=torch.long)
        z = torch.randn(n_per_digit, z_dim, device=DEVICE)
        rows.append(G(z, y).cpu())

    imgs = torch.cat(rows, dim=0)
    fig, axes = plt.subplots(NUM_CLASSES, n_per_digit, figsize=(n_per_digit, NUM_CLASSES))

    for row in range(NUM_CLASSES):
        for col in range(n_per_digit):
            ax = axes[row, col]
            ax.imshow(imgs[row * n_per_digit + col, 0], cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(str(row), rotation=0, labelpad=12, fontsize=12)

    plt.tight_layout()
    plt.show()


sample_digit_grid(G, z_dim=z_dim, n_per_digit=10, seed=0)

In [ ]:
@torch.no_grad()
def sample_digit_grid_with_preds(G, classifier, z_dim=64, n_per_digit=10, seed=None):
    G.eval()
    classifier.eval()
    if seed is not None:
        torch.manual_seed(seed)

    imgs = []
    ys = []
    for digit in range(NUM_CLASSES):
        y = torch.full((n_per_digit,), digit, device=DEVICE, dtype=torch.long)
        z = torch.randn(n_per_digit, z_dim, device=DEVICE)
        imgs.append(G(z, y))
        ys.append(y)

    imgs = torch.cat(imgs, dim=0)
    ys = torch.cat(ys, dim=0)
    preds = classifier(imgs).argmax(dim=1).cpu()

    fig, axes = plt.subplots(NUM_CLASSES, n_per_digit, figsize=(n_per_digit, NUM_CLASSES))
    for row in range(NUM_CLASSES):
        for col in range(n_per_digit):
            idx = row * n_per_digit + col
            ax = axes[row, col]
            ax.imshow(imgs[idx, 0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
            ax.set_title(str(int(preds[idx])), fontsize=8)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"y={row}", rotation=0, labelpad=14, fontsize=10)

    plt.tight_layout()
    plt.show()


sample_digit_grid_with_preds(G, classifier, z_dim=z_dim, n_per_digit=10, seed=1)

In [ ]:
def generated_covs_from_ema(ema_stats, layer):
    mu = ema_stats[layer]["mu"]
    M2 = ema_stats[layer]["M2"]
    covs = M2 - torch.einsum("bi,bj->bij", mu, mu)
    covs = 0.5 * (covs + covs.transpose(-1, -2))
    return covs


def plot_real_vs_generated_cov(layer="late", digit=0):
    real_cov = real_stats[layer]["cov"][digit].detach().cpu().numpy()
    gen_cov = generated_covs_from_ema(ema_stats, layer)[digit].detach().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, cov, title in zip(axes, [real_cov, gen_cov], ["real", "generated EMA"]):
        im = ax.imshow(cov, cmap="viridis")
        ax.set_title(f"{title} | {layer} | digit {digit}")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.show()


plot_real_vs_generated_cov(layer="early", digit=0)
plot_real_vs_generated_cov(layer="middle", digit=0)
plot_real_vs_generated_cov(layer="late", digit=0)